<a href="https://colab.research.google.com/github/cgm2179/indoor-walk-test/blob/main/SIM%20V2/phase_c_v2_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase C v2 — Train the 7-class enhanced-MK surrogate (sim-v2.0)

A **complete, runnable** fork of `phase_c_train_colab_v3.ipynb` for the v2
dataset. The learner is unchanged from v1 (DECISIONS D2); what differs is the
input (**10 channels**: 7-class one-hot + tx + freq + dist), the **v2 manifest**
(7 classes, [40, 230] dB clip window, 9-band freq range), and the **v2 baseline
evaluator** (uses `engine_v2.py`, not the v1 `phase_a.py`).

**Runtime → GPU.** Requires the v2 dataset from `phase_b_v2_generate_colab.ipynb`
(default Drive folder `MyDrive/SIM/dataset_v2`).

> The single most important v2 fix vs the old scaffold: this reads
> **`SIM V2/manifest_v2.json`**, not the v1 `SIM/manifest.json`. The v2 dataset
> was normalised with the [40, 230] dB window; using the v1 [40, 170] window
> here would mis-scale every target and quietly wreck training.

## 0. Repo + v2 manifest

In [1]:
import os
REPO = 'github.com/cgm2179/indoor-walk-test.git'
if not os.path.exists('indoor-walk-test'):
    if os.system(f'git clone --depth 1 https://{REPO}') != 0:
        from getpass import getpass
        token = getpass('GitHub fine-grained token: ')
        assert os.system(f'git clone --depth 1 https://{token}@{REPO}') == 0, 'clone failed'
%cd indoor-walk-test
import torch, json, numpy as np
print('GPU:', torch.cuda.is_available())

# v2 manifest — NOT the v1 one.  This carries the 7-class grid shape, the
# [40, 170] clip window (after the effective-obstruction calibration), and the 619–6125 MHz freq range.
manifest = json.load(open('SIM V2/manifest_v2.json'))
NORM = manifest['norm']; PL_LO, PL_RNG = NORM['pl_min_db'], NORM['pl_range_db']
H, W = manifest['grid_shape']; SIGMA = manifest['tx_blob_sigma_cells']
CELL = manifest['cell_size_m']; DN = manifest.get('dist_channel_norm', 3.0)
IN_CH = manifest.get('in_channels', manifest.get('in_ch'))   # 10
N_CLASSES = manifest['n_classes']        # 7
assert IN_CH == 10 and N_CLASSES == 7, (IN_CH, N_CLASSES)
print('grid', (H, W), '| clip', PL_LO, '-', PL_LO+PL_RNG, 'dB |',
      IN_CH, 'channels |', N_CLASSES, 'classes')
print('freqs:', manifest['freqs_mhz'])


/content/indoor-walk-test
GPU: True
grid (256, 448) | clip 40.0 - 170.0 dB | 10 channels | 7 classes
freqs: [619.0, 627.0, 1935.0, 2442.0, 2510.0, 2600.0, 3500.0, 5500.0, 6125.0]


## 1. Load the v2 dataset

Same shard format as v1 (`tx_pos, freq_feat, target, jitter, pos_id`), so this
loader is the v1 one pointed at the v2 folder.

In [2]:
from pathlib import Path
DATASET = Path('SIM V2/dataset_v2')
if not any(DATASET.glob('shard_*.npz')):
    from google.colab import drive
    drive.mount('/content/drive')
    root = Path('/content/drive/MyDrive')
    candidates = [root/'SIM'/'dataset_v2', root/'dataset_v2',
                  root/'indoor-walk-test-dataset-v2']
    candidates += [q for p in root.iterdir() if p.is_dir()
                   for q in [p] + [d for d in p.iterdir() if d.is_dir()]]
    DATASET = next((c for c in candidates if any(c.glob('shard_*.npz'))), None)
    assert DATASET, 'No v2 shard_*.npz found. Run phase_b_v2_generate_colab first.'
print('dataset from:', DATASET)

# the dataset was built against a specific grid; the manifest sha is the
# tripwire that catches a grid/dataset mismatch before it corrupts training
# tolerant grid-sha tripwire: accept either meta filename/key, or splits.json
def _ds_grid_sha(ds):
    for fn in ('dataset_meta_v2.json', 'dataset_v2_meta.json'):
        if (ds/fn).exists():
            mm = json.load(open(ds/fn))
            return mm.get('manifest_grid_sha') or mm.get('grid_sha256')
    return json.load(open(ds/'splits.json')).get('grid_sha256')
assert _ds_grid_sha(DATASET) == manifest['grid_sha256'], 'DATASET/GRID MISMATCH'
print('grid sha matches manifest:', manifest['grid_sha256'])

SAVE_DIR = (DATASET.parent/'checkpoints'
            if str(DATASET).startswith('/content/drive') else Path('.'))
SAVE_DIR.mkdir(exist_ok=True); print('checkpoints ->', SAVE_DIR)

splits = json.load(open(DATASET/'splits.json'))
shards = sorted(DATASET.glob('shard_*.npz'))
print(f'{len(shards)} shards, loading...')
tx_all, ff_all, tg_all, pid_all = [], [], [], []
for s in shards:
    d = np.load(s)
    tx_all.append(d['tx_pos']); ff_all.append(d['freq_feat'])
    tg_all.append(d['target']); pid_all.append(d['pos_id'])
TX = np.concatenate(tx_all); FF = np.concatenate(ff_all)
TG = np.concatenate(tg_all); PID = np.concatenate(pid_all)
print(f'{len(TG)} samples, {TG.nbytes/1e9:.2f} GB in RAM')
idx = {k: np.nonzero(np.isin(PID, splits[k]))[0] for k in ('train','val','test')}
print({k: len(v) for k, v in idx.items()})


Mounted at /content/drive
dataset from: /content/drive/MyDrive/SIM/dataset_v2
grid sha matches manifest: 0831cc022e542ea7
checkpoints -> /content/drive/MyDrive/SIM/checkpoints
20 shards, loading...
22500 samples, 5.16 GB in RAM
{'train': 18000, 'val': 2250, 'test': 2250}


## 2. Input builder — 7-class one-hot (10 channels)

The channel order MUST match `manifest['channels']`: 7 one-hot (grid class 0..6)
+ tx blob + freq + log-distance. This same logic must be mirrored in
`simulator_tab.js` when the v2 model ships to the dashboard.

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

grid = np.load('SIM V2/grid_model_v2.npy')
INSIDE = np.load('SIM V2/inside_mask_v2.npy')
onehot = np.stack([(grid == c) for c in range(N_CLASSES)]).astype(np.float32)
yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)

# sanity: the manifest channel order is exactly what we build here
_expect = [f'onehot_{m["name"]}' for m in manifest['materials']] + \
          ['tx_gaussian_sigma2', 'freq_feature', 'log10_distance_m_over3']
assert manifest['channels'] == _expect, 'channel order drift vs manifest!'

def build_input(tx_x, tx_y, ff):
    x = np.empty((IN_CH, H, W), np.float32)
    x[:N_CLASSES] = onehot
    d = np.hypot(xx - tx_x, yy - tx_y)
    x[N_CLASSES]     = np.exp(-d**2 / (2*SIGMA**2))       # tx blob
    x[N_CLASSES + 1] = ff                                  # freq feature
    x[N_CLASSES + 2] = np.log10(np.maximum(d*CELL, 1.0))/DN
    return x

class PLData(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        j = self.ids[i]
        return (torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j])),
                torch.from_numpy(TG[j].astype(np.float32)))

def loaders(batch):
    return (DataLoader(PLData(idx['train']), batch_size=batch, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True),
            DataLoader(PLData(idx['val']), batch_size=batch, num_workers=2))

MASK = torch.from_numpy(INSIDE.astype(np.float32))
print('input builder ready:', (IN_CH, H, W))


input builder ready: (10, 256, 448)


## 3. UNet (unchanged from v1 except in_ch=10)

In [4]:
import torch.nn as nn

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class UNet(nn.Module):
    def __init__(self, base=64, in_ch=IN_CH):
        super().__init__()
        ch = [base, base*2, base*4, base*8]
        self.enc = nn.ModuleList(); prev = in_ch
        for c in ch: self.enc.append(block(prev, c)); prev = c
        self.pool = nn.MaxPool2d(2)
        self.mid = block(prev, prev*2)
        self.up, self.dec = nn.ModuleList(), nn.ModuleList(); prev = prev*2
        for c in reversed(ch):
            self.up.append(nn.ConvTranspose2d(prev, c, 2, 2))
            self.dec.append(block(2*c, c)); prev = c
        self.head = nn.Conv2d(prev, 1, 1)          # R1: one output (path loss)
    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.mid(x)
        for u, d in zip(self.up, self.dec):
            x = d(torch.cat([u(x), skips.pop()], 1))
        return self.head(x)[:, 0]

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{sum(p.numel() for p in UNet().parameters())/1e6:.1f} M params, in_ch={IN_CH}')


31.0 M params, in_ch=10


## 4. Training loop (masked MSE + gradient-L1, Drive-resumable)

Identical to v1: AMP, Adam 1e-3 → cosine 1e-5, batch 16, ≤150 epochs, early
stop patience 15, full state checkpointed to Drive every 2 epochs so a Colab
disconnect resumes exactly. Checkpoint names carry `v2` so they never collide
with v1 runs.

In [5]:
import time, math
GRAD_W = 0.1
RESUME_EVERY = 2

def grad_l1(a, b):
    return ((a[:,1:,:]-a[:,:-1,:]) - (b[:,1:,:]-b[:,:-1,:])).abs().mean() + \
           ((a[:,:,1:]-a[:,:,:-1]) - (b[:,:,1:]-b[:,:,:-1])).abs().mean()

def ckpt(seed):        return SAVE_DIR / f'unet10_v2_seed{seed}_best.pt'
def resume_file(seed): return SAVE_DIR / f'unet10_v2_seed{seed}_resume.pt'

def train_one(seed, base=64, lr=1e-3, batch=16, epochs=150, patience=15):
    torch.manual_seed(seed); np.random.seed(seed)
    net = UNet(base).to(dev)
    opt = torch.optim.Adam(net.parameters(), lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler(dev)
    start_ep, best, best_ep = 0, 1e9, -1
    if resume_file(seed).exists():
        st = torch.load(resume_file(seed), map_location=dev)
        net.load_state_dict(st['net']); opt.load_state_dict(st['opt'])
        sched.load_state_dict(st['sched']); scaler.load_state_dict(st['scaler'])
        start_ep, best, best_ep = st['epoch']+1, st['best'], st['best_ep']
        print(f'RESUMED seed {seed} at epoch {start_ep+1}, best {best:.2f} dB')
    tr_dl, va_dl = loaders(batch)
    mask = MASK.to(dev); msum = mask.sum()
    for ep in range(start_ep, epochs):
        net.train(); t0 = time.time()
        for x, y in tr_dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.amp.autocast(dev):
                p = net(x)
                mse = (((p - y)**2) * mask).sum() / (msum * x.size(0))
                loss = mse + GRAD_W * grad_l1(p*mask, y*mask)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()
        net.eval(); se = n = 0
        with torch.no_grad():
            for x, y in va_dl:
                p = net(x.to(dev)).cpu()
                se += (((p - y)**2) * MASK).sum().item()
                n += MASK.sum().item() * y.size(0)
        rmse_db = PL_RNG * math.sqrt(se / n)
        if rmse_db < best:
            best, best_ep = rmse_db, ep
            torch.save(net.state_dict(), ckpt(seed))
        if ep % RESUME_EVERY == 0 or ep == epochs-1:
            torch.save(dict(net=net.state_dict(), opt=opt.state_dict(),
                            sched=sched.state_dict(), scaler=scaler.state_dict(),
                            epoch=ep, best=best, best_ep=best_ep), resume_file(seed))
        print(f'seed {seed} ep {ep+1:3d}  val RMSE {rmse_db:5.2f} dB (indoor)  '
              f'best {best:5.2f}  ({time.time()-t0:.0f}s)')
        if ep - best_ep >= patience:
            print(f'early stop (patience {patience})')
            resume_file(seed).unlink(missing_ok=True); break
    return best


## 5. Train

`QUICK=True` runs one seed for the first pass. Epoch 1 should already be in the
low tens of dB and fall fast; if it stalls above ~20 dB for several epochs,
something regressed (check the distance channel and the mask made it in). Set
`QUICK=False` for the 3-seed protocol once the result looks right.

In [ ]:
QUICK = True
SEEDS = [0] if QUICK else [0, 1, 2]
results = {s: train_one(s) for s in SEEDS}
vals = list(results.values())
print(f'val RMSE over {len(SEEDS)} seed(s): {np.mean(vals):.2f} ± {np.std(vals):.2f} dB')


RESUMED seed 0 at epoch 4, best 5.98 dB
seed 0 ep   4  val RMSE  5.20 dB (indoor)  best  5.20  (114s)
seed 0 ep   5  val RMSE  5.60 dB (indoor)  best  5.20  (102s)
seed 0 ep   6  val RMSE  5.05 dB (indoor)  best  5.05  (102s)
seed 0 ep   7  val RMSE  4.80 dB (indoor)  best  4.80  (102s)
seed 0 ep   8  val RMSE  4.72 dB (indoor)  best  4.72  (102s)
seed 0 ep   9  val RMSE  4.60 dB (indoor)  best  4.60  (102s)
seed 0 ep  10  val RMSE  4.74 dB (indoor)  best  4.60  (101s)
seed 0 ep  11  val RMSE  4.40 dB (indoor)  best  4.40  (102s)
seed 0 ep  12  val RMSE  4.45 dB (indoor)  best  4.40  (101s)
seed 0 ep  13  val RMSE  4.21 dB (indoor)  best  4.21  (102s)
seed 0 ep  14  val RMSE  4.22 dB (indoor)  best  4.21  (101s)
seed 0 ep  15  val RMSE  4.09 dB (indoor)  best  4.09  (102s)
seed 0 ep  16  val RMSE  4.11 dB (indoor)  best  4.09  (101s)
seed 0 ep  17  val RMSE  4.12 dB (indoor)  best  4.09  (102s)
seed 0 ep  18  val RMSE  3.95 dB (indoor)  best  3.95  (101s)
seed 0 ep  19  val RMSE  3.85 

## 6. Baselines (v2 engine) + evaluation

The baselines answer "is the surrogate actually learning the floor plan, or
could a map-blind model do as well?". FSPL and fitted log-distance are map-blind;
3GPP InH needs a LOS/NLOS mask, which we get from the **v2 engine** (`engine_v2`)
rather than the v1 `phase_a.py`. The surrogate must beat all three by a wide
margin.

In [ ]:
import sys
sys.path.insert(0, 'SIM')
import engine_v2 as E, physics_v2 as P

F_LO, F_HI = np.log10(NORM['freq_log_lo_mhz']), np.log10(NORM['freq_log_hi_mhz'])
dist_m = lambda j: np.maximum(np.hypot(xx - TX[j,0], yy - TX[j,1]) * CELL, 1.0)
f_of   = lambda j: 10 ** (FF[j] * (F_HI - F_LO) + F_LO)
ins = INSIDE

def fspl_map(j):
    return 32.44 + 20*np.log10(f_of(j)) - 60 + 20*np.log10(dist_m(j))

# fit the log-distance exponent on 400 training maps (same as v1)
sub = np.random.default_rng(0).choice(idx['train'], 400, replace=False)
num = den = 0.0
for j in sub:
    pl = TG[j].astype(np.float32) * PL_RNG + PL_LO
    ex = pl - (32.44 + 20*np.log10(f_of(j)) - 60)
    ld = 10*np.log10(dist_m(j))
    num += (ex[ins]*ld[ins]).sum(); den += (ld[ins]**2).sum()
N_FIT = num/den
print(f'fitted log-distance n = {N_FIT:.2f}')

def logdist_map(j):
    return 32.44 + 20*np.log10(f_of(j)) - 60 + 10*N_FIT*np.log10(dist_m(j))

def inh38901_map(j, los):
    d = dist_m(j); fghz = f_of(j)/1000
    pl_los = 32.4 + 17.3*np.log10(d) + 20*np.log10(fghz)
    pl_nlos = np.maximum(pl_los, 17.3 + 38.3*np.log10(d) + 24.9*np.log10(fghz))
    return np.where(los, pl_los, pl_nlos)

# a single v2 SceneV2 supplies LOS masks (zero obstruction along the ray).
# n_edges/relays 0: LOS only needs the direct obstruction, not diffraction.
_scene = E.SceneV2(grid, INSIDE, CELL, freqs_mhz=[3500.0],
                   n_edges=0, precompute=False)

def eval_maps(ids, model=None, kind='unet', los_cache=None):
    se = ae = be = n = 0
    for j in ids:
        gt = TG[j].astype(np.float32)*PL_RNG + PL_LO
        if kind == 'unet':
            xin = torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j]))[None]
            with torch.no_grad():
                p = model(xin.to(dev))[0].cpu().numpy()
            pr = np.clip(p, 0, 1)*PL_RNG + PL_LO
        elif kind == 'fspl':    pr = np.clip(fspl_map(j), PL_LO, PL_LO+PL_RNG)
        elif kind == 'logdist': pr = np.clip(logdist_map(j), PL_LO, PL_LO+PL_RNG)
        elif kind == '38901':
            key = tuple(TX[j])
            if key not in los_cache:
                obs = _scene.obstruction_maps((float(TX[j,0]), float(TX[j,1])))
                los_cache[key] = obs[0] == 0            # LOS = no obstruction
            pr = np.clip(inh38901_map(j, los_cache[key]), PL_LO, PL_LO+PL_RNG)
        e = (pr - gt)[ins]
        se += (e**2).sum(); ae += np.abs(e).sum(); be += e.sum(); n += e.size
    return dict(rmse=np.sqrt(se/n), mae=ae/n, bias=be/n)

net = UNet().to(dev)
net.load_state_dict(torch.load(ckpt(SEEDS[0]), map_location=dev)); net.eval()
val_sub = np.random.default_rng(1).choice(idx['val'], 200, replace=False)
los_cache = {}
for kind in ('fspl', 'logdist', '38901', 'unet'):
    r = eval_maps(val_sub, net, kind, los_cache)
    print(f"VAL {kind:8s} RMSE {r['rmse']:6.2f}  MAE {r['mae']:6.2f}  bias {r['bias']:+6.2f} dB")


## 7. Test set (touch once) + qualitative maps

In [ ]:
test_sub = idx['test']
for kind in ('fspl', 'logdist', '38901', 'unet'):
    r = eval_maps(test_sub, net, kind, los_cache)
    print(f"TEST {kind:8s} RMSE {r['rmse']:6.2f}  MAE {r['mae']:6.2f}  bias {r['bias']:+6.2f} dB")


In [ ]:
import matplotlib.pyplot as plt
CLIP_HI = PL_LO + PL_RNG
js = idx['test'][:3]
fig, axes = plt.subplots(3, 3, figsize=(16, 7.5), squeeze=False)
for r, j in enumerate(js):
    gt = TG[j].astype(np.float32)*PL_RNG + PL_LO
    xin = torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j]))[None]
    with torch.no_grad():
        p = net(xin.to(dev))[0].cpu().numpy()
    pr = np.clip(p, 0, 1)*PL_RNG + PL_LO
    for c, (img, ttl, cm, lo, hi) in enumerate([
            (gt, 'v2 simulated PL (dB)', 'turbo', 40, CLIP_HI),
            (pr, 'UNet prediction', 'turbo', 40, CLIP_HI),
            (np.abs(pr-gt), '|error| (dB)', 'magma', 0, 10)]):
        a = axes[r][c]
        im = a.imshow(np.where(ins, img, np.nan), cmap=cm, vmin=lo, vmax=hi)
        a.set_title(ttl if r == 0 else '', fontsize=10); a.axis('off')
        fig.colorbar(im, ax=a, shrink=0.7)
plt.tight_layout()


## 8. D.3 physical sanity checks

In [ ]:
j = idx['test'][0]
xin = torch.from_numpy(build_input(TX[j,0], TX[j,1], FF[j]))[None]
with torch.no_grad():
    p = net(xin.to(dev))[0].cpu().numpy()*PL_RNG + PL_LO
fx, fy = TX[j]
ray = p[int(fy), int(fx):int(fx)+40]
print('monotone LOS decay (fraction increasing):', (np.diff(ray) > -0.5).mean().round(2))
print('cells below FSPL:', f'{100*(p < fspl_map(j) - 1)[ins].mean():.2f}% (want ~0)')
x24 = build_input(TX[j,0], TX[j,1], 0.0)          # ff=0 -> lowest band (619 MHz)
with torch.no_grad():
    p24 = net(torch.from_numpy(x24)[None].to(dev))[0].cpu().numpy()*PL_RNG + PL_LO
print('median(PL@619MHz <= PL@current):', f'{np.median((p24 <= p + 0.5)[ins]):.0f} (want 1)')


## 9. Export to ONNX (single file, parity gate ≤ 0.1 dB)

Same export as v1, with the two guards that caught real failures: assert the
file is a single self-contained >50 MB file (not the dynamo graph+data split),
and assert ONNX↔torch parity ≤ 0.1 dB before shipping. Downloads
`pl_unet.onnx` → put it at `SIM/web/pl_unet.onnx` on your machine (and bump the
`?v=N` cache-buster).

In [ ]:
!pip -q install onnx onnxscript onnxruntime
import os, onnx
net.cpu().eval()
try:
    torch.onnx.export(net, torch.zeros(1, IN_CH, H, W), 'pl_unet_v2.onnx',
                      input_names=['x'], output_names=['pl_norm'],
                      opset_version=17, dynamo=False)
except TypeError:
    torch.onnx.export(net, torch.zeros(1, IN_CH, H, W), 'pl_unet_v2_ext.onnx',
                      input_names=['x'], output_names=['pl_norm'], opset_version=17)
    onnx.save(onnx.load('pl_unet_v2_ext.onnx'), 'pl_unet_v2.onnx',
              save_as_external_data=False)
size_mb = os.path.getsize('pl_unet_v2.onnx')/1e6
print(f'file size: {size_mb:.1f} MB')
assert size_mb > 50, 'weights NOT embedded (two-file export) — do not ship'
import onnxruntime as ort_rt
sess = ort_rt.InferenceSession('pl_unet_v2.onnx')
worst = 0
for j in idx['test'][:50]:
    xin = build_input(TX[j,0], TX[j,1], FF[j])
    with torch.no_grad():
        pt = net(torch.from_numpy(xin)[None])[0].numpy()
    ox = sess.run(None, {sess.get_inputs()[0].name: xin[None]})[0][0]
    worst = max(worst, np.abs(pt - ox).max()*PL_RNG)
print(f'ONNX parity: max |diff| {worst:.4f} dB (must be <= 0.1)')
assert worst <= 0.1, 'PARITY FAIL — do not ship this file'
net.to(dev)
import shutil
if str(SAVE_DIR).startswith('/content/drive'):
    shutil.copy('pl_unet_v2.onnx', SAVE_DIR/'pl_unet_v2.onnx')
    print('copied to', SAVE_DIR/'pl_unet_v2.onnx')
from google.colab import files
files.download('pl_unet_v2.onnx')


## 10. Acceptance (MODEL_CARD_V2 §12)

- v2 beats FSPL / fitted-log-distance / 3GPP-InH baselines by a wide margin.
- Compared with v1's 4.68 dB test RMSE: a v2 physics improvement can legitimately
  raise the *sim*-RMSE while lowering *field* RMSE — document which.
- ONNX single-file, parity ≤ 0.1 dB.
- Keep this release for the physics-ladder ablation (multiwall vs enhanced-MK
  vs calibrated). Deploy as `surrogate-v2` + `dataset-v2`.

Every constant here is literature (P.2040 + fitted excess). Phase D calibrates
to the building once a known-Tx walk exists; the `jitter` field makes that a
fine-tune, not a retrain.